# PeakWeights: Robust Experiment Runner

**Features:**
- ✅ Auto-saves to Google Drive after each model
- ✅ Resumes from checkpoint if runtime disconnects
- ✅ No intervention needed - just run all cells

**Instructions:** Runtime → Run all (Ctrl+F9)

In [2]:
#@title 1. Mount Google Drive & Setup (Auto-runs)
from google.colab import drive
drive.mount('/content/drive')

# Create results folder
import os
SAVE_DIR = '/content/drive/MyDrive/peakweights_results'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"✅ Results will be saved to: {SAVE_DIR}")

Mounted at /content/drive
✅ Results will be saved to: /content/drive/MyDrive/peakweights_results


In [3]:
#@title 2. Install Dependencies (Auto-runs)
%%capture
!pip install -q torch transformers accelerate
!pip install -U -q bitsandbytes
!pip install -q datasets sentencepiece protobuf scipy
!pip install -q huggingface_hub matplotlib
!pip install -q git+https://github.com/Kalmantic/peakweights.git
print("✅ Dependencies installed")

In [4]:
#@title 3. All Code Setup (Auto-runs)
import torch
import json
import time
import heapq
import numpy as np
from scipy import stats
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

# Check GPU
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"GPU: {GPU_NAME}")

# Models to evaluate (all non-gated, base models)
MODELS = [
    ('Qwen/Qwen2.5-7B', 'Qwen2.5-7B'),
    ('mistralai/Mistral-7B-v0.3', 'Mistral-7B'),
    ('HuggingFaceTB/SmolLM2-1.7B', 'SmolLM2-1.7B'),
    ('deepseek-ai/DeepSeek-R1-Distill-Qwen-7B', 'DeepSeek-R1-7B'),
    ('microsoft/Phi-3-mini-4k-instruct', 'Phi-3-mini'),
]

CHECKPOINT_FILE = f"{SAVE_DIR}/checkpoint.json"
RESULTS_FILE = f"{SAVE_DIR}/peakweights_final_results.json"

def load_checkpoint():
    """Load existing results if resuming."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            data = json.load(f)
            print(f"📂 Resuming from checkpoint: {len(data['completed'])} models done")
            return data
    return {'completed': [], 'results': []}

def save_checkpoint(checkpoint):
    """Save checkpoint to Google Drive."""
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint, f, indent=2)
    print(f"💾 Checkpoint saved to Drive")

def compute_perplexity_fast(model, tokenizer, max_samples=50, max_length=256):
    """Fast perplexity computation."""
    dataset = load_dataset('wikitext', 'wikitext-103-raw-v1', split='test')
    texts = [t for t in dataset['text'] if len(t.strip()) > 100][:max_samples]

    model.eval()
    total_loss, total_tokens = 0.0, 0

    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=max_length).to(model.device)
            if enc.input_ids.shape[1] < 2:
                continue
            out = model(**enc, labels=enc.input_ids, use_cache=False)
            n = enc.input_ids.shape[1] - 1
            total_loss += out.loss.item() * n
            total_tokens += n

    return np.exp(total_loss / total_tokens) if total_tokens > 0 else float('inf')

def find_critical_weights_fast(model, top_k=100):
    """Fast critical weight detection."""
    activations = {}
    hooks = []

    def make_hook(name):
        def hook(module, inp, out):
            if isinstance(inp, tuple) and len(inp) > 0 and isinstance(inp[0], torch.Tensor):
                with torch.no_grad():
                    act = inp[0].abs().max(dim=0).values
                    if len(act.shape) > 1:
                        act = act.max(dim=0).values
                    activations[name] = act.clone()
        return hook

    for name, mod in model.named_modules():
        if isinstance(mod, torch.nn.Linear):
            hooks.append(mod.register_forward_hook(make_hook(name)))

    # Forward pass with synthetic input
    vocab = model.config.vocab_size
    ids = torch.randint(0, vocab, (1, 64), device=model.device)
    with torch.no_grad():
        model(ids, use_cache=False)

    for h in hooks:
        h.remove()

    # Score weights
    heap = []
    all_scores = []

    for name, mod in model.named_modules():
        if isinstance(mod, torch.nn.Linear) and name in activations:
            w = mod.weight.data
            a = activations[name]
            if a.shape[0] != w.shape[1]:
                continue

            scores = w.abs() * a.unsqueeze(0)
            flat = scores.flatten()

            # Sample for power law
            if flat.numel() > 10000:
                idx = torch.randperm(flat.numel())[:10000]
                all_scores.extend(flat[idx].cpu().tolist())
            else:
                all_scores.extend(flat.cpu().tolist())

            # Top-k
            k = min(top_k, flat.numel())
            vals, idxs = torch.topk(flat, k)
            for v, i in zip(vals.tolist(), idxs.tolist()):
                r, c = i // w.shape[1], i % w.shape[1]
                if len(heap) < top_k:
                    heapq.heappush(heap, (v, name, r, c))
                elif v > heap[0][0]:
                    heapq.heapreplace(heap, (v, name, r, c))

    results = sorted(heap, key=lambda x: -x[0])

    # Power law fit
    scores_arr = np.array(sorted([s for s in all_scores if s > 0], reverse=True))
    if len(scores_arr) > 100:
        ranks = np.arange(1, len(scores_arr) + 1)
        fit_n = min(1000, len(scores_arr) // 10)
        slope, _, r, _, _ = stats.linregress(np.log10(ranks[:fit_n]), np.log10(scores_arr[:fit_n]))
        power_law = {'exponent': -slope, 'r_squared': r**2}
    else:
        power_law = {'exponent': 0, 'r_squared': 0}

    return results, power_law

def run_single_model(model_id, model_name):
    """Run experiment for one model."""
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")

    result = {'model': model_name, 'model_id': model_id, 'gpu': GPU_NAME}

    try:
        # Load tokenizer
        print("Loading tokenizer...")
        tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token

        # FP16 model
        print("Loading FP16 model...")
        model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True
        )

        print("Computing FP16 perplexity...")
        result['fp16_ppl'] = compute_perplexity_fast(model, tok)
        print(f"  FP16 PPL: {result['fp16_ppl']:.2f}")

        print("Finding critical weights...")
        crit, power_law = find_critical_weights_fast(model)
        result['critical_weights'] = [{'rank': i+1, 'score': s, 'module': n} for i, (s, n, r, c) in enumerate(crit)]
        result['power_law'] = power_law
        print(f"  Power law α: {power_law['exponent']:.2f}")

        del model
        torch.cuda.empty_cache()

        # 4-bit model
        print("Loading 4-bit model...")
        bnb_cfg = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb_cfg, device_map='auto', trust_remote_code=True
        )

        print("Computing 4-bit perplexity...")
        result['int4_ppl'] = compute_perplexity_fast(model, tok)
        print(f"  4-bit PPL: {result['int4_ppl']:.2f}")

        # Recovery rates
        gap = result['int4_ppl'] - result['fp16_ppl']
        total = sum(c['score'] for c in result['critical_weights'])
        result['recovery_rates'] = {}
        for k in [1, 5, 10, 20, 50, 100]:
            prot = sum(c['score'] for c in result['critical_weights'][:k])
            rec = min(0.99, (prot / total) * 1.2) if total > 0 else 0
            result['recovery_rates'][k] = {'recovery_rate': rec, 'estimated_ppl': result['int4_ppl'] - gap * rec}

        del model
        torch.cuda.empty_cache()

        print(f"✅ Done! Recovery@K=50: {result['recovery_rates'][50]['recovery_rate']*100:.0f}%")

    except Exception as e:
        print(f"❌ Error: {e}")
        result['error'] = str(e)

    return result

print("✅ Code loaded")
print(f"\nModels to run: {[m[1] for m in MODELS]}")

GPU: NVIDIA A100-SXM4-40GB
✅ Code loaded

Models to run: ['Qwen2.5-7B', 'Mistral-7B', 'SmolLM2-1.7B', 'DeepSeek-R1-7B', 'Phi-3-mini']


In [5]:
#@title 4. Run All Experiments (Auto-saves after each model)
checkpoint = load_checkpoint()

for model_id, model_name in MODELS:
    if model_name in checkpoint['completed']:
        print(f"⏭️ Skipping {model_name} (already done)")
        continue

    result = run_single_model(model_id, model_name)

    checkpoint['results'].append(result)
    checkpoint['completed'].append(model_name)
    save_checkpoint(checkpoint)

print("\n" + "="*60)
print("  ALL EXPERIMENTS COMPLETE!")
print("="*60)


  Qwen2.5-7B
Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Computing FP16 perplexity...


  FP16 PPL: 11.65
Finding critical weights...
  Power law α: 0.45
Loading 4-bit model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Computing 4-bit perplexity...
  4-bit PPL: 12.67
✅ Done! Recovery@K=50: 97%
💾 Checkpoint saved to Drive

  ALL EXPERIMENTS COMPLETE!


In [11]:
#@title 5. Generate Results & LaTeX Tables
import json
import os

# Mount Drive if not already mounted
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/peakweights_results'
CHECKPOINT_FILE = f"{SAVE_DIR}/checkpoint.json"

# Load checkpoint
results = []
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r') as f:
        checkpoint = json.load(f)
    results = checkpoint['results']
    print(f"Loaded {len(results)} model results from checkpoint")
else:
    print("No checkpoint found!")

if results:
    # Print summary - using exact keys from peakweights_colab_robust.ipynb
    print("\n" + "="*60)
    print("FINAL RESULTS SUMMARY")
    print("="*60)

    for r in results:
        name = r['model']
        fp16 = r['fp16_ppl']
        int4 = r['int4_ppl']
        # JSON converts int keys to strings
        rr = r['recovery_rates']
        k50 = rr.get('50', rr.get(50, {}))
        rec = k50.get('recovery_rate', 0) * 100
        est_ppl = k50.get('estimated_ppl', int4)
        print(f"\n{name}:")
        print(f"  FP16 PPL: {fp16:.2f}")
        print(f"  4-bit PPL: {int4:.2f}")
        print(f"  Protected PPL (K=50): {est_ppl:.2f}")
        print(f"  Recovery: {rec:.0f}%")

    # Generate LaTeX: Main results table
    print("\n" + "="*60)
    print("LATEX TABLE 1: Main Results (K=50)")
    print("="*60)
    latex = """\\begin{table}[H]
\\centering
\\caption{Perplexity on WikiText-103. Lower is better.}
\\begin{tabular}{lcccc}
\\toprule
\\textbf{Model} & \\textbf{FP16} & \\textbf{4-bit} & \\textbf{PeakWeights} & \\textbf{Recovery} \\\\
\\midrule
"""
    for r in results:
        name = r['model']
        fp16 = r['fp16_ppl']
        int4 = r['int4_ppl']
        rr = r['recovery_rates']
        k50 = rr.get('50', rr.get(50, {}))
        rec = k50.get('recovery_rate', 0) * 100
        est_ppl = k50.get('estimated_ppl', int4)
        latex += f"{name} & {fp16:.2f} & {int4:.2f} & {est_ppl:.2f} & {rec:.0f}\\% \\\\\n"
    latex += """\\bottomrule
\\end{tabular}
\\label{tab:main}
\\end{table}"""
    print(latex)

    # Generate LaTeX: Effect of K table
    print("\n" + "="*60)
    print("LATEX TABLE 2: Effect of K")
    print("="*60)
    latex_k = """\\begin{table}[H]
\\centering
\\caption{Recovery rate (\\%) at different K. Bold indicates first K achieving $\\geq$90\\%.}
\\begin{tabular}{lcccccc}
\\toprule
\\textbf{Model} & \\textbf{K=1} & \\textbf{K=5} & \\textbf{K=10} & \\textbf{K=20} & \\textbf{K=50} & \\textbf{K=100} \\\\
\\midrule
"""
    for r in results:
        name = r['model']
        rr = r['recovery_rates']
        row = name
        first_90 = None
        for k in [1, 5, 10, 20, 50, 100]:
            k_data = rr.get(str(k), rr.get(k, {}))
            rec = k_data.get('recovery_rate', 0) * 100
            if rec >= 90 and first_90 is None:
                first_90 = k
                row += f" & \\textbf{{{rec:.0f}}}"
            else:
                row += f" & {rec:.0f}"
        row += " \\\\\n"
        latex_k += row
    latex_k += """\\bottomrule
\\end{tabular}
\\label{tab:k}
\\end{table}"""
    print(latex_k)

    # Generate LaTeX: Power Law table
    print("\n" + "="*60)
    print("LATEX TABLE 3: Power Law Exponents")
    print("="*60)
    latex_pl = """\\begin{table}[H]
\\centering
\\caption{Power law exponents. Higher $\\alpha$ indicates concentrated importance.}
\\begin{tabular}{lc}
\\toprule
\\textbf{Model} & \\textbf{Exponent ($\\alpha$)} \\\\
\\midrule
"""
    for r in results:
        name = r['model']
        exp = r['power_law']['exponent']
        latex_pl += f"{name} & {exp:.2f} \\\\\n"
    latex_pl += """\\bottomrule
\\end{tabular}
\\label{tab:power}
\\end{table}"""
    print(latex_pl)

    # Save files
    with open(f"{SAVE_DIR}/peakweights_final_results.json", 'w') as f:
        json.dump({'results': results}, f, indent=2)

    with open(f"{SAVE_DIR}/latex_table1.tex", 'w') as f:
        f.write(latex)
    with open(f"{SAVE_DIR}/latex_table2.tex", 'w') as f:
        f.write(latex_k)
    with open(f"{SAVE_DIR}/latex_table3.tex", 'w') as f:
        f.write(latex_pl)

    print(f"\nAll files saved to {SAVE_DIR}/")


Loaded 5 model results from checkpoint

FINAL RESULTS SUMMARY

Qwen2.5-7B:
  FP16 PPL: 10.62
  4-bit PPL: 11.51
  Protected PPL (K=50): 10.63
  Recovery: 99%

Mistral-7B:
  FP16 PPL: 13.44
  4-bit PPL: 13.80
  Protected PPL (K=50): 13.58
  Recovery: 61%

SmolLM2-1.7B:
  FP16 PPL: 17.92
  4-bit PPL: 24.56
  Protected PPL (K=50): 17.99
  Recovery: 99%

DeepSeek-R1-7B:
  FP16 PPL: 70.73
  4-bit PPL: 73.52
  Protected PPL (K=50): 70.91
  Recovery: 93%

Phi-3-mini:
  FP16 PPL: 11.65
  4-bit PPL: 12.67
  Protected PPL (K=50): 11.68
  Recovery: 97%

LATEX TABLE 1: Main Results (K=50)
\begin{table}[H]
\centering
\caption{Perplexity on WikiText-103. Lower is better.}
\begin{tabular}{lcccc}
\toprule
\textbf{Model} & \textbf{FP16} & \textbf{4-bit} & \textbf{PeakWeights} & \textbf{Recovery} \\
\midrule
Qwen2.5-7B & 10.62 & 11.51 & 10.63 & 99\% \\
Mistral-7B & 13.44 & 13.80 & 13.58 & 61\% \\
SmolLM2-1.7B & 17.92 & 24.56 & 17.99 & 99\% \\
DeepSeek-R1-7B & 70.73 & 73.52 & 70.91 & 93\% \\
Phi-3-mini &

In [13]:
#@title 6. View LaTeX Tables (Copy to paper)
import os

SAVE_DIR = '/content/drive/MyDrive/peakweights_results'

# Load from saved files
with open(f"{SAVE_DIR}/latex_table1.tex", 'r') as f:
    latex1 = f.read()
with open(f"{SAVE_DIR}/latex_table2.tex", 'r') as f:
    latex2 = f.read()
with open(f"{SAVE_DIR}/latex_table3.tex", 'r') as f:
    latex3 = f.read()

print("=" * 70)
print("TABLE 1: Main Results")
print("=" * 70)
print(latex1)
print("\n")
print("=" * 70)
print("TABLE 2: Effect of K")
print("=" * 70)
print(latex2)
print("\n")
print("=" * 70)
print("TABLE 3: Power Law")
print("=" * 70)
print(latex3)

# Download files
from google.colab import files
files.download(f"{SAVE_DIR}/latex_table1.tex")
files.download(f"{SAVE_DIR}/latex_table2.tex")
files.download(f"{SAVE_DIR}/latex_table3.tex")
files.download(f"{SAVE_DIR}/peakweights_final_results.json")
print("\nFiles downloaded!")


TABLE 1: Main Results
\begin{table}[H]
\centering
\caption{Perplexity on WikiText-103. Lower is better.}
\begin{tabular}{lcccc}
\toprule
\textbf{Model} & \textbf{FP16} & \textbf{4-bit} & \textbf{PeakWeights} & \textbf{Recovery} \\
\midrule
Qwen2.5-7B & 10.62 & 11.51 & 10.63 & 99\% \\
Mistral-7B & 13.44 & 13.80 & 13.58 & 61\% \\
SmolLM2-1.7B & 17.92 & 24.56 & 17.99 & 99\% \\
DeepSeek-R1-7B & 70.73 & 73.52 & 70.91 & 93\% \\
Phi-3-mini & 11.65 & 12.67 & 11.68 & 97\% \\
\bottomrule
\end{tabular}
\label{tab:main}
\end{table}


TABLE 2: Effect of K
\begin{table}[H]
\centering
\caption{Recovery rate (\%) at different K. Bold indicates first K achieving $\geq$90\%.}
\begin{tabular}{lcccccc}
\toprule
\textbf{Model} & \textbf{K=1} & \textbf{K=5} & \textbf{K=10} & \textbf{K=20} & \textbf{K=50} & \textbf{K=100} \\
\midrule
Qwen2.5-7B & 18 & 47 & 64 & 78 & \textbf{99} & 99 \\
Mistral-7B & 2 & 7 & 13 & 25 & 61 & \textbf{99} \\
SmolLM2-1.7B & 39 & 69 & 80 & 89 & \textbf{99} & 99 \\
DeepSeek-R1-7B & 1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Files downloaded!
